In [ ]:
# === Setup cell — run once per session ===
!apt-get install -y openjdk-17-jre-headless -qq > /dev/null
!pip install mpxj openpyxl JPype1 -q

In [ ]:
#!/usr/bin/env python3
"""
Convert a Microsoft Project .mpp file into the Project for the Web
"Excel Import Template" (Project / Resources / Tasks sheets), with the
Tasks data wrapped in an Excel table named "Estimates".

IMPORTANT — why this fills an embedded template instead of building a
workbook from scratch:
    The SQIT import plugin's Excel reader reads INLINE strings only. A
    workbook built fresh by openpyxl stores text in a shared-strings table
    (cells hold only integer indices), which the plugin reads as blank —
    the symptom is validation reporting "Project Name missing" and
    "Tasks: 0" even though the file clearly has data. Loading and filling
    the shipped template keeps every string cell as t="inlineStr", which
    the plugin reads correctly. Do not switch this back to openpyxl.Workbook().

Requirements:
    pip install mpxj openpyxl jpype1
"""

import sys
import glob
import os
import io
import base64
import datetime
import re

# ---------------------------------------------------------------------------
# Embedded empty import template (inline-string .xlsx). Filled at runtime.
# ---------------------------------------------------------------------------
_TEMPLATE_B64 = """
...
AAAMAAwAEAMAAOhJAAAAAA==
"""

def _load_template():
    raw = base64.b64decode("".join(_TEMPLATE_B64.split()))
    return io.BytesIO(raw)

# ---------------------------------------------------------------------------
# Start the JVM with the mpxj jars on the classpath
# ---------------------------------------------------------------------------
import jpype

if not jpype.isJVMStarted():
    try:
        import mpxj
        lib_dir = os.path.join(os.path.dirname(mpxj.__file__), "lib")
        jars = glob.glob(os.path.join(lib_dir, "*.jar"))
    except Exception:
        jars = []
    if not jars:
        raise RuntimeError("Could not locate mpxj jars. Install with: pip install mpxj")
    jpype.startJVM(classpath=jars)

UniversalProjectReader = jpype.JClass("org.mpxj.reader.UniversalProjectReader")

import openpyxl
from openpyxl.worksheet.table import Table, TableStyleInfo

HOURS_PER_DAY = 8

# Authoritative active resource roles (bookableresourcecategory names) exactly as
# they exist in Dataverse. The plugin matches role names case-insensitively but
# otherwise exactly, so these strings are what must land in the workbook.
CANONICAL_ROLES = [
    "L1-BP", "L1-CARD PERSO", "L1-IT", "L1-MANAGED SERVICE", "L1-NANO-PERSO",
    "L1-NANO-SWITCH", "L1-PMO", "L1-SOFTWARE DEV", "L1-SUPPORT",
    "L2-BP", "L2-CARD PERSO", "L2-IT", "L2-MANAGED SERVICE", "L2-NANO-PERSO",
    "L2-NANO-SWITCH", "L2-PMO", "L2-Senior QC Manager", "L2-SOFTWARE DEV", "L2-SUPPORT",
    "L3-BP", "L3-CARD PERSO", "L3-IT", "L3-MANAGED SERVICE", "L3-NANO-PERSO",
    "L3-NANO-SWITCH", "L3- PMO", "L3-SOFTWARE DEV", "L3-SUPPORT",
    "L4-BP", "L4-CARD PERSO", "L4-IT", "L4-MANAGED SERVICE", "L4-NANO-PERSO",
    "L4-NANO-SWITCH", "L4-PMO", "L4-SOFTWARE DEV", "L4-SUPPORT",
    "External", "Project Manager",
]

# Discipline spelling variants -> the canonical discipline token used in the role
# names above. Lets long forms ("Software Development") and abbreviations resolve
# to the exact Dataverse spelling ("SOFTWARE DEV").
_DISCIPLINE_ALIASES = {
    "BP": ["BP", "BUSINESS PARTNER", "BUSINESS PROCESS"],
    "CARD PERSO": ["CARD PERSO", "CARD PERSONALIZATION", "CARD PERSONALISATION", "CARDPERSO"],
    "IT": ["IT", "INFORMATION TECHNOLOGY"],
    "MANAGED SERVICE": ["MANAGED SERVICE", "MANAGED SERVICES", "MS"],
    "NANO-PERSO": ["NANO PERSO", "NANO PERSONALIZATION", "NANO PERSONALISATION", "NANOPERSO", "NANO-PERSO"],
    "NANO-SWITCH": ["NANO SWITCH", "NANOSWITCH", "NANO-SWITCH", "NS"],
    "PMO": ["PMO", "PROJECT MANAGEMENT", "PROJECT MANAGEMENT OFFICE"],
    "SOFTWARE DEV": ["SOFTWARE DEV", "SOFTWARE DEVELOPMENT", "SOFTWARE DEVELOPER",
                     "SW DEV", "SWDEV", "DEV", "SD"],
    "SUPPORT": ["SUPPORT", "SUPP"],
    "SENIOR QC MANAGER": ["SENIOR QC MANAGER", "SR QC MANAGER", "SENIOR QC MGR", "QC MANAGER"],
}


def _norm(s):
    """Uppercase; _, -, / -> space; collapse whitespace. Matching only."""
    s = s.upper().replace("_", " ")
    s = re.sub(r"[-/]", " ", s)
    return re.sub(r"\s+", " ", s).strip()


_ALIAS_TO_DISCIPLINE = {}
for _disc, _aliases in _DISCIPLINE_ALIASES.items():
    for _a in _aliases:
        _ALIAS_TO_DISCIPLINE[_norm(_a)] = _disc

# (level, canonical-discipline) -> exact Dataverse role string.
_ROLE_BY_KEY = {}
for _role in CANONICAL_ROLES:
    _m = re.match(r"^L\s*([1-4])\s*-?\s*(.*)$", _role, re.I)
    if not _m:
        continue
    _dn = _norm(_m.group(2))
    _disc = _ALIAS_TO_DISCIPLINE.get(_dn, _dn)
    _ROLE_BY_KEY[(_m.group(1), _disc)] = _role


def _canonical_role(name):
    """
    Resolve an L<1-4> label to its exact Dataverse role string, else None.
    Handles any spacing/case, "-"/"/" separators, and discipline spelling
    variants (e.g. "L3 - Software Development" -> "L3-SOFTWARE DEV").
    """
    if name is None:
        return None
    m = re.match(r"^\s*L\s*([1-4])\b[\s\-/]*(.*)$", str(name).strip(), re.I)
    if not m:
        return None
    rest = _norm(m.group(2))
    if not rest:
        return None
    disc = _ALIAS_TO_DISCIPLINE.get(rest, rest)
    return _ROLE_BY_KEY.get((m.group(1), disc))


def format_resource_name(resource_name):
    """
    Canonical label for BOTH the Resources sheet 'Resource Name' column and the
    Tasks sheet 'Resources' column, so they always agree. An L<1-4> role becomes
    its exact Dataverse spelling; anything else is returned unchanged.
    """
    if resource_name is None or str(resource_name).strip() == "":
        return resource_name
    canon = _canonical_role(resource_name)
    return canon if canon is not None else str(resource_name).strip()


def map_role(resource_name):
    """
    Value for the Resources sheet 'Role' column. An L<1-4> role maps to its exact
    Dataverse role string; anything else -> 'External' (which is a real role).
    """
    if resource_name is None or str(resource_name).strip() == "":
        return None
    canon = _canonical_role(resource_name)
    return canon if canon is not None else "External"


def read_project(mpp_path):
    return UniversalProjectReader().read(mpp_path)


def work_hours(task):
    w = task.getWork()
    if w is None:
        return 0.0
    try:
        TimeUnit = jpype.JClass("org.mpxj.TimeUnit")
        return float(w.convertUnits(TimeUnit.HOURS, None).getDuration())
    except Exception:
        try:
            return float(w.getDuration())
        except Exception:
            return 0.0


def duration_days(task):
    d = task.getDuration()
    if d is None:
        return None
    try:
        TimeUnit = jpype.JClass("org.mpxj.TimeUnit")
        return float(d.convertUnits(TimeUnit.DAYS, None).getDuration())
    except Exception:
        try:
            return float(d.getDuration())
        except Exception:
            return None


def is_milestone(task):
    try:
        return bool(task.getMilestone())
    except Exception:
        return False


def resource_names(task):
    out = []
    for a in task.getResourceAssignments():
        r = a.getResource()
        if r is not None and r.getName() is not None:
            n = format_resource_name(str(r.getName()).strip())
            if n and n not in out:
                out.append(n)
    return out


def percent_complete(task):
    try:
        p = task.getPercentageComplete()
        if p is None:
            return None
        return int(round(float(p)))
    except Exception:
        return None


def fmt_date(dt):
    """
    Convert an mpxj date/datetime to a Python datetime, preserving time-of-day
    so Start/Finish match what MS Project shows (e.g. 8:00 AM / 5:00 PM).
    Returns None when absent. Written as a real Excel datetime and formatted
    MPP-style at write time.
    """
    if dt is None:
        return None
    s = str(dt).strip()
    if not s:
        return None
    # Normalise a trailing "Z" and space/T separator, then let fromisoformat try.
    iso = s.replace("Z", "").replace(" ", "T")
    try:
        return datetime.datetime.fromisoformat(iso)
    except ValueError:
        pass
    for fmt in ("%Y-%m-%dT%H:%M:%S", "%Y-%m-%dT%H:%M", "%Y-%m-%d"):
        try:
            return datetime.datetime.strptime(iso[:19], fmt)
        except ValueError:
            continue
    try:
        return datetime.datetime.strptime(s[:10], "%Y-%m-%d")
    except ValueError:
        return None


# ---------------------------------------------------------------------------
# Flatten the task tree
# ---------------------------------------------------------------------------
def build_rows(project):
    tasks = [t for t in project.getTasks()
             if t is not None and t.getName() is not None]

    def has_children(t):
        c = t.getChildTasks()
        return c is not None and not c.isEmpty()

    rows = []
    id_map = {}
    resource_order = []
    counter = [1]
    pending_preds = {}

    def add_resource(n):
        if n not in resource_order:
            resource_order.append(n)

    def emit(t, parent_template_id):
        this_id = counter[0]
        counter[0] += 1
        id_map[int(t.getID())] = this_id

        summary = has_children(t)
        row = {
            "Task ID": this_id,
            "Parent Task ID": parent_template_id,
            "Task Name": str(t.getName())[:300],
            "Start": None, "Duration (days)": None, "Finish": None,
            "Effort (hours)": None, "% Complete": None,
            "Predecessors": None, "Resources": None, "Notes": None,
            "Estimates Start Date": None,
        }

        if not summary:
            row["Start"] = fmt_date(t.getStart())
            row["Finish"] = fmt_date(t.getFinish())
            if is_milestone(t):
                row["Duration (days)"] = 0
            else:
                dd = duration_days(t)
                if dd is None:
                    wh = work_hours(t)
                    dd = round(wh / HOURS_PER_DAY, 2) if wh else None
                row["Duration (days)"] = round(dd, 2) if dd is not None else None
            wh = work_hours(t)
            row["Effort (hours)"] = round(wh, 2) if wh else None
            row["% Complete"] = percent_complete(t)

            rns = resource_names(t)
            for n in rns:
                add_resource(n)
            row["Resources"] = "; ".join(rns) if rns else None

            preds = []
            try:
                for rel in t.getPredecessors():
                    src = rel.getPredecessorTask()
                    if src is not None:
                        preds.append(int(src.getID()))
            except Exception:
                pass
            if preds:
                pending_preds[this_id] = preds

        rows.append(row)
        if summary:
            for c in t.getChildTasks():
                if c is not None and c.getName() is not None:
                    emit(c, this_id)
        return this_id

    min_level = min(int(t.getOutlineLevel()) for t in tasks)
    roots = [t for t in tasks if int(t.getOutlineLevel()) == min_level]
    while len(roots) == 1 and has_children(roots[0]):
        roots = [c for c in roots[0].getChildTasks()
                 if c is not None and c.getName() is not None]
    for r in roots:
        emit(r, None)

    row_by_id = {r["Task ID"]: r for r in rows}
    for tid, raw_preds in pending_preds.items():
        mapped = [str(id_map[p]) for p in raw_preds if p in id_map]
        if mapped:
            row_by_id[tid]["Predecessors"] = "; ".join(mapped)

    # Project overall start = earliest task start across the schedule. Stamp the
    # same value onto every row's "Estimates Start Date" column.
    starts = [r["Start"] for r in rows if r["Start"] is not None]
    project_start = min(starts) if starts else None
    if project_start is not None:
        for r in rows:
            r["Estimates Start Date"] = project_start

    return rows, resource_order


# ---------------------------------------------------------------------------
# Fill the embedded template (keeps inline strings -> plugin can read it)
# ---------------------------------------------------------------------------
def write_excel(rows, resources, out_path, project_name="Imported Project"):
    wb = openpyxl.load_workbook(_load_template())

    # --- Project: set the Project Name value (row with Setting == Project Name)
    ws_p = wb["Project"]
    for r in range(2, ws_p.max_row + 1):
        if str(ws_p.cell(r, 1).value).strip().lower() == "project name":
            ws_p.cell(r, 2, project_name)
            break

    # --- Resources
    ws_r = wb["Resources"]
    # clear anything below header just in case
    for r in range(2, ws_r.max_row + 1):
        for c in range(1, 4):
            ws_r.cell(r, c, None)
    for i, name in enumerate(resources, start=2):
        ws_r.cell(i, 1, name)
        ws_r.cell(i, 2, None)            # blank -> generic team member
        ws_r.cell(i, 3, map_role(name))  # preserved role mapping

    # --- Tasks
    ws_t = wb["Tasks"]
    headers = ["Task ID", "Parent Task ID", "Task Name", "Start",
               "Duration (days)", "Finish", "Effort (hours)", "% Complete",
               "Predecessors", "Resources", "Notes", "Estimates Start Date"]
    # The embedded template's Tasks header row only covers the 11 built-in
    # columns (A-K). Write the added header into row 1 so column L has a proper
    # header cell (an Excel table rejects empty header cells).
    ws_t.cell(1, len(headers), "Estimates Start Date")
    # clear old rows
    for r in range(2, ws_t.max_row + 1):
        for c in range(1, len(headers) + 1):
            ws_t.cell(r, c, None)

    r = 2
    for row in rows:
        for c, h in enumerate(headers, start=1):
            ws_t.cell(r, c, row[h])
        # MPP-style display: e.g. "Mon 6/1/26 8:00 AM". Underlying value is a
        # real Excel datetime, so it stays sortable/computable.
        _mpp_fmt = "ddd m/d/yy h:mm AM/PM"
        if row["Start"] is not None:
            ws_t.cell(r, 4).number_format = _mpp_fmt
        if row["Finish"] is not None:
            ws_t.cell(r, 6).number_format = _mpp_fmt
        if row["Estimates Start Date"] is not None:
            ws_t.cell(r, 12).number_format = _mpp_fmt
        r += 1
    last_row = max(r - 1, 2)

    # Remove any pre-existing table, then add the Estimates table over the data
    try:
        if "Estimates" in ws_t.tables:
            del ws_t.tables["Estimates"]
    except Exception:
        pass
    table = Table(displayName="Estimates", ref="A1:L{}".format(last_row))
    table.tableStyleInfo = TableStyleInfo(
        name="TableStyleMedium2", showFirstColumn=False,
        showLastColumn=False, showRowStripes=True, showColumnStripes=False)
    ws_t.add_table(table)

    wb.save(out_path)


# ---------------------------------------------------------------------------
def main():
    try:
        from google.colab import files  # type: ignore
        in_colab = True
    except Exception:
        in_colab = False

    if in_colab:
        print("Please upload your .mpp file...")
        uploaded = files.upload()
        if not uploaded:
            print("No file uploaded. Aborting.")
            return
        mpp_path = list(uploaded.keys())[0]
        base = os.path.splitext(os.path.basename(mpp_path))[0]
        out_path = base + "_Import.xlsx"

        project = read_project(mpp_path)
        rows, resources = build_rows(project)
        write_excel(rows, resources, out_path, project_name=base)
        print("Wrote {} task rows, {} resources -> {}".format(len(rows), len(resources), out_path))
        files.download(out_path)
        return

    if len(sys.argv) < 2:
        print("Usage: python mpp_to_import.py <input.mpp> [output.xlsx]")
        sys.exit(1)
    mpp_path = sys.argv[1]
    base = os.path.splitext(os.path.basename(mpp_path))[0]
    out_path = sys.argv[2] if len(sys.argv) >= 3 else base + "_Import.xlsx"
    project = read_project(mpp_path)
    rows, resources = build_rows(project)
    write_excel(rows, resources, out_path, project_name=base)
    print("Wrote {} task rows, {} resources -> {}".format(len(rows), len(resources), out_path))


if __name__ == "__main__":
    main()